# Phase B — Cross-model dynamic speculative decoding analysis

Analyses the 33 Phase A trace cells (Llama 18/18 + Qwen 15/18, missing only Qwen v3_dynamic × 3) to answer:

1. **Where does static EAGLE3 waste compute?** — per-position rejection rates
2. **Which signal (DOG = top1×RAR, or t = 0.7·top1 + 0.3·target) predicts AL better?** — binned discriminability
3. **Is one set of thresholds enough cross-model / cross-dataset?** — KS divergence of signal CDFs
4. **Where can dynamic spec decoding 'buy' throughput?** — overspend + underspend regions
5. **What thresholds to use in V3 (DOG_DIVISOR) / V6 (CHEAP, PREMIUM) for Phase C?** — grid search

Inputs:
- `results/traces/master.parquet` built by `analysis/phase_b_build_master.py`
- `results/traces/metrics_final.csv` for cross-checks

Outputs:
- `analysis/figures/plot_{1..5}_*.{png,pdf}`
- `analysis/phase_b_summary.md` — decision gate + recommendations

In [ ]:
# ── Section 0: imports, config, paths ─────────────────────────────────
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), 'analysis'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from phase_b_analysis import (
    load_master, cell_key,
    STATIC_CONFIGS, DYNAMIC_CONFIGS, ALL_CONFIGS, CONFIG_SHAPE,
    MODELS, DATASETS, SIGNALS,
    plot1_first_reject_cdf, plot1_wasted_tail_table,
    plot2_signal_deciles,  plot2_spearman,
    plot3_ks_matrix,       plot3_ks_summary,
    plot4_buy_signal,
    plot5_best_config_per_decile, plot5_derive_thresholds,
    phase_b_summary_md,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 90

REPO_ROOT   = '/rds/user/ou222/hpc-work/diss'
MASTER_PATH = os.path.join(REPO_ROOT, 'results/traces/master.parquet')
FIG_DIR     = os.path.join(REPO_ROOT, 'analysis/figures')
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    for ext in ('png', 'pdf'):
        plt.savefig(os.path.join(FIG_DIR, f'{name}.{ext}'),
                    bbox_inches='tight', dpi=150 if ext == 'png' else 300)
    plt.show()

In [ ]:
# ── Section 1: load master dataframe ─────────────────────────────────
master = load_master(MASTER_PATH)
print(f'Master df: {len(master):,} rows × {len(master.columns)} cols')
master.head()

In [ ]:
# Coverage matrix — sanity check before going further
coverage = (
    master.groupby(['model', 'dataset', 'config'], observed=True)
          .size().unstack(['config']).fillna(0).astype(int)
)
coverage

## Plot 1 — Where static makes mistakes (first-reject position CDFs)

For each static config × (model, dataset), we plot the empirical CDF of `first_reject_pos` — the first draft-tree position that verification rejected. A CDF that climbs sharply at low positions = chain runs out early. A CDF flat for many positions = those positions rarely fail.

`first_reject_pos = num_steps` (rightmost point) means the full chain was accepted that step.

In [ ]:
cdfs = plot1_first_reject_cdf(master)

fig, axes = plt.subplots(
    len(MODELS), len(DATASETS), figsize=(15, 8),
    sharex=False, sharey=True,
)
config_colors = {c: col for c, col in
                 zip(STATIC_CONFIGS, sns.color_palette('tab10', n_colors=4))}

for i, mdl in enumerate(MODELS):
    for j, dset in enumerate(DATASETS):
        ax = axes[i, j]
        for cfg in STATIC_CONFIGS:
            key = (cfg, mdl, dset)
            if key not in cdfs:
                continue
            d = cdfs[key]
            ax.step(d['position'], d['cum_fraction'], where='post',
                    label=f'{cfg} (ns={CONFIG_SHAPE[cfg][0]})',
                    color=config_colors[cfg], linewidth=2)
        ax.set_title(f'{mdl} / {dset}')
        ax.set_xlabel('first rejected position')
        if j == 0:
            ax.set_ylabel('cumulative fraction of steps')
        ax.set_ylim(-0.02, 1.02)
        ax.grid(True, alpha=0.3)
        if i == 0 and j == 0:
            ax.legend(loc='lower right', fontsize=8)

plt.suptitle('Plot 1 — CDF of first-rejected draft position per static config',
             y=1.02, fontsize=13)
plt.tight_layout()
savefig('plot_1_first_reject_cdf')

In [ ]:
# Wasted tail table — fraction of steps where first_reject_pos <= num_steps/2
wasted = plot1_wasted_tail_table(master)
wasted.style.background_gradient(
    subset=['wasted_tail_fraction'], cmap='Reds'
).format({'wasted_tail_fraction': '{:.1%}', 'mean_al': '{:.2f}'})

**Interpretation**: a `wasted_tail_fraction` above 50% means that more than half of the draft steps reject in the first half of the chain — the deeper positions are almost never reached. Those configs over-spend draft compute.

The lower the fraction, the more the chain depth is actually paying off. A very low fraction on a long chain (e.g. `static_7_1_8` with wasted_tail < 20%) would argue for deeper chains; a high fraction suggests reverting to a shorter chain.

## Plot 2 — Signal discriminability (DOG, t vs AL)

Per-decile mean `accept_length` plotted against the signal value. A steeper, more monotonic curve means the signal more reliably predicts whether an aggressive config will pay off.

Spearman rank correlation quantifies this per cell × config.

In [ ]:
def plot_signal_to_al(signal_name, label):
    tab = plot2_signal_deciles(master, signal_name)
    fig, axes = plt.subplots(
        len(MODELS), len(DATASETS), figsize=(15, 8),
        sharex=True, sharey=True,
    )
    config_colors = {c: col for c, col in
                     zip(STATIC_CONFIGS, sns.color_palette('tab10', n_colors=4))}
    for i, mdl in enumerate(MODELS):
        for j, dset in enumerate(DATASETS):
            ax = axes[i, j]
            sub = tab[(tab['model'] == mdl) & (tab['dataset'] == dset)]
            for cfg in STATIC_CONFIGS:
                s = sub[sub['config'] == cfg].sort_values('decile')
                if len(s) == 0:
                    continue
                ax.plot(s['signal_mid'], s['mean_al'], 'o-',
                        color=config_colors[cfg], label=cfg, markersize=5)
                ax.fill_between(s['signal_mid'], s['ci_lo'], s['ci_hi'],
                                color=config_colors[cfg], alpha=0.15)
            ax.set_title(f'{mdl} / {dset}')
            if i == 1: ax.set_xlabel(label)
            if j == 0: ax.set_ylabel('mean accept_length')
            if i == 0 and j == 0:
                ax.legend(fontsize=7, loc='upper left')
    plt.suptitle(f'Plot 2 — {label} (deciles) vs mean accept_length',
                 y=1.02, fontsize=13)
    plt.tight_layout()
    return tab

tab_dog = plot_signal_to_al('DOG', 'DOG = top1 × RAR (V3 signal)')
savefig('plot_2a_DOG_vs_AL')

In [ ]:
tab_t = plot_signal_to_al('t', 't = 0.7·top1 + 0.3·target_top1 (V6 signal)')
savefig('plot_2b_t_vs_AL')

In [ ]:
# Spearman rank correlation per cell × config
spearman_dog = plot2_spearman(master, 'DOG')
spearman_t   = plot2_spearman(master, 't')

print('DOG Spearman (median across cells×configs):', 
      f"{spearman_dog['spearman_r'].median():.3f}")
print('  t Spearman (median across cells×configs):',
      f"{spearman_t['spearman_r'].median():.3f}")

sp_summary = (
    pd.concat([spearman_dog.assign(signal='DOG'),
               spearman_t.assign(signal='t')])
    .pivot_table(index=['model', 'dataset', 'config'],
                 columns='signal', values='spearman_r')
    .reset_index()
)
sp_summary.style.background_gradient(subset=['DOG', 't'], cmap='Greens', axis=None)

**Interpretation**:
- Higher Spearman r = signal better predicts per-step AL.
- Compare DOG vs t — whichever has higher median across cells is the better scalar for policy decisions.
- Per-cell r values far from median = non-stationary signal; V6's cross-model constants may not hold.

## Plot 3 — Cross-cell CDF overlap + KS divergence

For each signal, overlay empirical CDFs from the 6 (model, dataset) cells using static-only runs (dynamic runs bias the signal distribution via `chosen_*`).

If CDFs stack tightly → one global threshold works. If they diverge → per-model or adaptive thresholds needed.

Decision gate (from plan):
- max KS < 0.10 → global thresholds OK
- 0.10 ≤ max KS < 0.20 → adaptive quantiles recommended
- max KS ≥ 0.20 → per-model thresholds required

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
palette = sns.color_palette('husl', n_colors=6)

cells = [(m, d) for m in MODELS for d in DATASETS]
cell_colors = {c: col for c, col in zip(cells, palette)}

for idx, signal in enumerate(SIGNALS):
    ax = axes.flat[idx]
    static_sub = master[master['config'].isin(STATIC_CONFIGS)]
    for mdl in MODELS:
        for dset in DATASETS:
            vals = static_sub[(static_sub['model'] == mdl) &
                              (static_sub['dataset'] == dset)][signal].to_numpy()
            if len(vals) == 0: continue
            sorted_vals = np.sort(vals)
            cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
            ax.plot(sorted_vals, cdf, color=cell_colors[(mdl, dset)],
                    label=f'{mdl}/{dset}', alpha=0.9, linewidth=1.8)
    ax.set_title(signal)
    ax.set_xlabel('signal value')
    ax.set_ylabel('CDF')
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.3)
    if idx == 0:
        ax.legend(loc='lower right', fontsize=8)

axes.flat[-1].axis('off')
plt.suptitle('Plot 3 — Cross-cell empirical CDFs per signal (static-only)',
             y=1.00, fontsize=13)
plt.tight_layout()
savefig('plot_3_cdf_overlap')

In [ ]:
# KS matrices per signal
ks_mats = {sig: plot3_ks_matrix(master, sig) for sig in SIGNALS}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, (signal, mat) in enumerate(ks_mats.items()):
    ax = axes.flat[idx]
    sns.heatmap(mat.astype(float), annot=True, fmt='.2f',
                vmin=0, vmax=0.3, cmap='YlOrRd',
                cbar=False, ax=ax, square=True, linewidths=0.5)
    ax.set_title(f'KS({signal})')
axes.flat[-1].axis('off')
plt.suptitle('Plot 3b — Pairwise KS divergence per signal (static-only traces)',
             y=1.02, fontsize=13)
plt.tight_layout()
savefig('plot_3b_ks_matrices')

In [ ]:
ks_summary = plot3_ks_summary(ks_mats)
ks_summary.style.background_gradient(
    subset=['max_ks'], cmap='Reds'
).format({'max_ks': '{:.3f}', 'mean_ks': '{:.3f}',
          'max_ks_within_model': '{:.3f}',
          'max_ks_across_model': '{:.3f}'})

**Interpretation**: low `max_ks` = signal distribution is stationary cross-cell. A single threshold works.

Compare `max_ks_within_model` (same model, different dataset) vs `max_ks_across_model` (different model, same dataset). If cross-model is much larger → per-model thresholds; if cross-dataset is larger → the global "one policy" assumption is shakier and per-dataset branching becomes a consideration despite being declared out-of-scope.

## Plot 4 — Where dynamic can 'buy' throughput

Per signal decile per cell, measure two counterfactual metrics from static runs:

- **Overspend** = `mean((ndt - accept_length) * [ndt ≥ 8])` — how many verified-but-unused tree positions the bigger configs spend.
- **Underspend** = `mean(accept_length ≥ num_steps − 1)` — fraction of steps where the small chain maxed out (could have gone deeper).

The "buy zone" is where **both** are notable — that's where dynamic switching has the biggest gain.

In [ ]:
def plot_buy(signal, label):
    buy = plot4_buy_signal(master, signal)
    fig, axes = plt.subplots(len(MODELS), len(DATASETS),
                              figsize=(15, 8), sharex=True)
    for i, mdl in enumerate(MODELS):
        for j, dset in enumerate(DATASETS):
            ax = axes[i, j]
            sub = buy[(buy['model'] == mdl) & (buy['dataset'] == dset)]
            agg = (sub.groupby('signal_mid', as_index=False)
                   .agg(overspend=('overspend', 'max'),
                        underspend=('underspend', 'max'))
                   .sort_values('signal_mid'))
            if len(agg) == 0: continue
            ax.plot(agg['signal_mid'], agg['overspend'], 'o-',
                    color='#d62728', label='overspend (big tree waste)')
            ax.plot(agg['signal_mid'], agg['underspend'], 's-',
                    color='#2ca02c', label='underspend (chain ceiling hit)')
            ax.set_title(f'{mdl} / {dset}')
            if i == 1: ax.set_xlabel(label)
            if j == 0: ax.set_ylabel('mean metric')
            if i == 0 and j == 0:
                ax.legend(fontsize=7)
    plt.suptitle(f'Plot 4 — Buy-zone analysis by {label}',
                 y=1.02, fontsize=13)
    plt.tight_layout()
    return buy

buy_dog = plot_buy('DOG', 'DOG decile mid')
savefig('plot_4a_buy_DOG')

In [ ]:
buy_t = plot_buy('t', 't decile mid')
savefig('plot_4b_buy_t')

**Interpretation**:

- **Overspend peak at low-signal decile + Underspend peak at high-signal decile** → the signal cleanly separates cheap vs expensive regimes; dynamic policy has a large lever.
- **Both peaks overlap (same decile)** → signal mixes cheap & expensive in the same bucket; dynamic gain will be limited.
- **Compare DOG vs t** — the signal with the cleaner separation wins.

## Plot 5 — Threshold recommendations

For each cell, find the signal-decile-wise best static config by `yield_per_compute = accept_length / draft_tree_size`. Then derive:

- **CHEAP threshold** = highest decile where `static_3_1_4` is best (switch to start config below)
- **PREMIUM threshold** = lowest decile where a big config is best (switch to max config above)

In [ ]:
best_per_decile_dog = plot5_best_config_per_decile(master, 'DOG')
best_per_decile_t   = plot5_best_config_per_decile(master, 't')

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
cfg_palette = {c: col for c, col in
               zip(STATIC_CONFIGS, sns.color_palette('tab10', 4))}

for ax_row, (sig_name, bpd) in enumerate([('DOG', best_per_decile_dog),
                                            ('t', best_per_decile_t)]):
    for j, dset in enumerate(DATASETS):
        ax = axes[ax_row, j]
        for mdl in MODELS:
            sub = bpd[(bpd['model'] == mdl) & (bpd['dataset'] == dset)]
            for _, row in sub.iterrows():
                ax.scatter(row['signal_mid'], mdl,
                           color=cfg_palette[row['best_config']],
                           s=100, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{sig_name} — {dset}')
        ax.set_xlabel(f'{sig_name} decile mid')
        if j == 0: ax.set_ylabel(f'model ({sig_name})')

# Legend
from matplotlib.patches import Patch
handles = [Patch(color=cfg_palette[c], label=c) for c in STATIC_CONFIGS]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.03))
plt.suptitle('Plot 5 — Best static config per signal decile per cell',
             y=1.00, fontsize=13)
plt.tight_layout()
savefig('plot_5a_best_config_per_decile')

In [ ]:
thresholds_dog = plot5_derive_thresholds(best_per_decile_dog)
thresholds_t   = plot5_derive_thresholds(best_per_decile_t)

print('DOG thresholds per cell:')
display(thresholds_dog)
print('\n t thresholds per cell:')
display(thresholds_t)

In [ ]:
# Global thresholds = median across the 6 per-cell thresholds
def global_from_cell(df, col):
    vals = df[col].dropna()
    return float(vals.median()) if len(vals) else float('nan')

global_DOG = {
    'cheap': global_from_cell(thresholds_dog, 'cheap_threshold'),
    'premium': global_from_cell(thresholds_dog, 'premium_threshold'),
}
global_t = {
    'cheap': global_from_cell(thresholds_t, 'cheap_threshold'),
    'premium': global_from_cell(thresholds_t, 'premium_threshold'),
}
print(f"DOG global thresholds (median per-cell):  cheap={global_DOG['cheap']:.3f}   premium={global_DOG['premium']:.3f}")
print(f"  t global thresholds (median per-cell):  cheap={global_t['cheap']:.3f}   premium={global_t['premium']:.3f}")
print()
print('V6 current constants (for reference):    cheap=0.30   premium=0.95')
print('V3 DOG_DIVISOR = 0.5 (t = DOG / 0.5, no zone threshold)')

## Summary — write `analysis/phase_b_summary.md`

In [ ]:
summary = phase_b_summary_md(
    ks_summary=ks_summary,
    spearman_dog=spearman_dog,
    spearman_t=spearman_t,
    thresholds_t=thresholds_t,
    thresholds_dog=thresholds_dog,
    wasted_tail=wasted,
)

summary_path = os.path.join(REPO_ROOT, 'analysis/phase_b_summary.md')
with open(summary_path, 'w') as f:
    f.write(summary)
print(f'Wrote {summary_path}')
print('\n' + summary[:2000])